# Gene Expression Modeling

## Classification: Normal vs Tumor

In this notebook, we build a machine learning model to classify samples based on gene expression profiles.

In [ ]:
#imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## Data Preparation

To build a machine learning model, we need to restructure the dataset so that each row represents a sample and each column represents a gene.

We also define a target variable indicating whether each sample corresponds to normal or tumor cells.

In [5]:
# Load data
df_counts_filtered = pd.read_csv('../data/counts_filtered.csv', index_col=0)

# CPM normalization
df_cpm = df_counts_filtered.div(df_counts_filtered.sum(axis=0), axis=1) * 1_000_000

# Log transform
df_log_cpm = np.log2(df_cpm + 1)

print("Data ready ✓")

Data ready ✓


### Reshaping the data

The original dataset is organized with genes as rows and samples as columns.  
For machine learning, we transpose the matrix so that samples become rows and genes become features.

In [6]:
# Transpose data
df_ml = df_log_cpm.T

df_ml.head()

Symbol,DDX11L1,WASH7P,MIR6859-1,FAM138A,LOC100996442,LOC729737,DDX11L17,WASH9P,MIR6859-2,LOC107985721,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
HPNE_Control,0.161907,3.027788,0.236520,0.042212,1.129381,0.236520,0.236520,3.524903,0.083224,0.000000,...,6.259157,0.725267,0.776179,0.750948,5.876503,2.855024,1.168020,6.694646,1.841223,2.567726
HPNE_H2O2,0.281593,3.377295,0.607338,0.043752,1.643660,0.281593,0.167568,3.974352,0.127465,0.043752,...,5.730593,0.966814,1.054975,1.011568,5.597473,4.083188,3.170829,5.685557,0.692265,1.643660
PANC1_Control,0.485003,3.658340,0.579884,0.027194,1.059769,0.155967,0.424927,3.874022,0.633955,0.000000,...,6.612539,0.616155,0.251301,0.180398,6.214843,3.440490,0.561400,6.069540,3.343761,4.775676
PANC1_H2O2,0.389656,3.948686,0.898244,0.000000,1.134895,0.235346,0.484061,4.223006,0.551020,0.000000,...,6.610300,0.551020,0.288638,0.314563,5.969004,3.245332,0.389656,5.984732,3.204319,5.156340


### Defining the target variable

We assign labels to each sample:
- HPNE → Normal cells  
- PANC1 → Tumor cells  

This will be the variable the model tries to predict.

In [7]:
# Create target variable
df_ml["label"] = ["HPNE", "HPNE", "PANC1", "PANC1"]

df_ml

Symbol,DDX11L1,WASH7P,MIR6859-1,FAM138A,LOC100996442,LOC729737,DDX11L17,WASH9P,MIR6859-2,LOC107985721,...,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP,label
HPNE_Control,0.161907,3.027788,0.236520,0.042212,1.129381,0.236520,0.236520,3.524903,0.083224,0.000000,...,0.725267,0.776179,0.750948,5.876503,2.855024,1.168020,6.694646,1.841223,2.567726,HPNE
HPNE_H2O2,0.281593,3.377295,0.607338,0.043752,1.643660,0.281593,0.167568,3.974352,0.127465,0.043752,...,0.966814,1.054975,1.011568,5.597473,4.083188,3.170829,5.685557,0.692265,1.643660,HPNE
PANC1_Control,0.485003,3.658340,0.579884,0.027194,1.059769,0.155967,0.424927,3.874022,0.633955,0.000000,...,0.616155,0.251301,0.180398,6.214843,3.440490,0.561400,6.069540,3.343761,4.775676,PANC1
PANC1_H2O2,0.389656,3.948686,0.898244,0.000000,1.134895,0.235346,0.484061,4.223006,0.551020,0.000000,...,0.551020,0.288638,0.314563,5.969004,3.245332,0.389656,5.984732,3.204319,5.156340,PANC1


## Feature Selection

Given the high dimensionality of gene expression data and the small number of samples, we select a subset of the most relevant genes.

We use the top differentially expressed genes identified in the previous analysis to reduce the feature space.

In [9]:
# Load differential expression results
df_de = pd.read_csv('../results/differential_expression.csv', index_col=0)

# Select top 50 genes by p-value
top_genes = df_de.sort_values(by="p_value").head(50).index

# Subset ML dataset
X = df_ml[top_genes]
y = df_ml["label"]

print("Selected features:", X.shape)

Selected features: (4, 50)


## Model Selection

Given the small sample size, a simple and interpretable model was selected.

Logistic Regression is appropriate in this context because:
- It performs well in high-dimensional settings
- It is less prone to overfitting compared to more complex models
- It allows straightforward interpretation of results

More complex models were not considered due to the limited number of samples.

## Model Training

Due to the extremely small dataset (4 samples), the model is trained using all available data.

This approach is intended for demonstration purposes, rather than building a production-ready model.

In [10]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = LogisticRegression()
model.fit(X_scaled, y)

print("Model trained ✓")

Model trained ✓


## Model Evaluation

The model is evaluated on the same data used for training.

While this is not a valid strategy for assessing real-world performance, it allows us to verify that the model can separate the classes under these conditions.

In [11]:
y_pred = model.predict(X_scaled)

print("Accuracy:", accuracy_score(y, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y, y_pred))
print("\nClassification Report:\n", classification_report(y, y_pred))

Accuracy: 1.0

Confusion Matrix:
 [[2 0]
 [0 2]]

Classification Report:
               precision    recall  f1-score   support

        HPNE       1.00      1.00      1.00         2
       PANC1       1.00      1.00      1.00         2

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4



## Interpretation of Results

The model achieves perfect accuracy on the training data.

This result is expected due to:
- The extremely small number of samples
- The use of the same data for training and evaluation
- The selection of highly informative features

This indicates overfitting, meaning the model has learned the training data rather than generalizable patterns.

In a real-world scenario, a larger dataset and proper validation strategy would be required.